Analysation of the data coming from the simulator.

In [ ]:
import yaml
import math
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from pathlib import Path, PurePath

In [ ]:
environment = "arena_perpendicular"

dataset_path = Path("wireless_channel/sionna_dataset/", environment)

# Read out the config file.
config_file = f"environments/{environment}/config.yaml"
with open(config_file, "r", encoding="utf8") as file:
    config = yaml.safe_load(file)

stripe_config = config["stripe_config"]

In [ ]:
def find_closest_ru(data, ue, n_stripes, n_rus):
    """
    data: nested list of dictionaries
    ue: dict with 'x', 'y', 'z' of the UE
    """

    ue_x, ue_y = ue["x"], ue["y"]

    closest_distance = float("inf")
    closest_index = None
    closest_ru_coords = None

    # Loop over outer list (groups)
    ru_index = 0
    for group in data:

        # Loop over elements inside each group
        for entry in group:
            if "radio_unit" in entry:  # skip central_unit
                ru = entry["radio_unit"]
                dx = ru["x"] - ue_x
                dy = ru["y"] - ue_y
                dist = math.sqrt(dx*dx + dy*dy)

                if dist < closest_distance:
                    closest_distance = dist
                    closest_index = ru_index
                    closest_ru_coords = (ru["x"], ru["y"])

                ru_index += 1
            
    closest_stripe = closest_index // n_rus
    closest_ru = closest_index % n_rus

    return closest_distance, closest_ru_coords, closest_stripe, closest_ru

def calculate_distance(ue, ru_coords, dstripe, dru, start):
    """Calculate the distance between the UE and the RU."""

    ue_x, ue_y = ue["x"], ue["y"]
    ru_x, ru_y = ru_coords[0], ru_coords[1]
    startx, starty = start

    dx = (ru_x * dstripe) + startx - ue_x
    dy = (ru_y * dru) + starty - ue_y
    dist = math.sqrt(dx*dx + dy*dy)

    return dist

In [ ]:
find_closest_ru(config['radio_stripes'], {'x': 9.0, 'y': 21.5}, 8, 20)

In [ ]:
file_path = PurePath(dataset_path, "ue_locations/ue_locations.nc")
ue_ds = xr.load_dataset(file_path)

rows = []
for ue_id in range(len(ue_ds['user_id'])):
    user = ue_ds.where(ue_ds["user_id"] == ue_id, drop=True)
    ue_x = float(user["x"])
    ue_y = float(user["y"])
    ue_z = float(user["z"])

    n_stripes = stripe_config['N_stripes']
    n_rus = stripe_config['N_RUs']
    dstripe = stripe_config['space_between_stripes']
    dru = stripe_config['space_between_RUs']
    startx, starty, startz = stripe_config['stripe_start_pos']
    dist, ccoords, stripe_idx, ru_idx = find_closest_ru(config['radio_stripes'], user, n_stripes, n_rus)

    # Read in the data for UEx
    try:
        file_path = PurePath(dataset_path, f"flickering/flickering_data_{ue_id}.pkl")
        df = pd.read_pickle(file_path)
    except FileNotFoundError:
        continue

    # For this UE just select the RU with the best NMSE and calculate the distance to that RU.
    row = df[df["nmse"] == df["nmse"].min()]
    row["cstripe_idx"] = stripe_idx
    row["cru_idx"] = ru_idx
    row["cru"] = dist
    dist = calculate_distance(user, (row["stripe_id"].iloc[0], row["ru_id"].iloc[0]), dstripe, dru, (startx, starty))
    row["dru"] = dist
    row["uex"] = ue_x
    row["uey"] = ue_y
    rows.append(row)

    resdf = pd.concat(rows)
    file_path = PurePath(dataset_path, "flickering/results.csv")
    resdf.to_csv(file_path, index=False)

In [ ]:
file_path = PurePath(dataset_path, "flickering/results.csv")
df = pd.read_csv(file_path)

fig, ax = plt.subplots()

# First plot all the RU positions on our figure.
x0 = stripe_config['stripe_start_pos'][0]
xend = stripe_config['stripe_end_pos'][0]
dx = stripe_config['space_between_stripes']
y0 = stripe_config['stripe_start_pos'][1]
yend = stripe_config['stripe_end_pos'][1]
dy = stripe_config['space_between_RUs']
n_rus = stripe_config['N_RUs']
n_stripes = stripe_config['N_stripes']

# Compute the RU positions.
x_pos = np.linspace(x0, xend, n_rus)
y_pos = np.linspace(y0, yend, n_rus)

# Repeat x_pos n_rus times, adding dx to each repetition.
x_pos_repeated = np.array([x_pos + i * dx for i in range(n_stripes)])
y_pos_repeated = np.array([y_pos for i in range(n_stripes)])

for sx, sy in zip(x_pos_repeated, y_pos_repeated):
    for x, y in zip(sx, sy):
        circle = Circle((y, x), radius=0.1, edgecolor='red', facecolor='none', linewidth=2)
        ax.add_patch(circle)

# Now create a grid and calculate the average NMSE for every grid window.
x_min = 0
x_max = config['room']["x"]
y_min = 0
y_max = config['room']["y"]

# Now make a heatmap of the NMSE.
# Define the grid dimensions
cell_size = 2  # 2x2 meter cells

# Create the grid
x_bins = np.arange(x_min, x_max + cell_size, cell_size)
y_bins = np.arange(y_min, y_max + cell_size, cell_size)

# Initialize the numpy array to store average NMSE values
grid = np.zeros((len(x_bins) - 1, len(y_bins) - 1))

# Iterate over the grid cells and calculate average NMSE
for i in range(len(x_bins) - 1):
    for j in range(len(y_bins) - 1):
        # Find UEs that fall into the current grid cell
        in_cell = df[(df["uex"] >= x_bins[i]) & (df["uex"] < x_bins[i + 1]) &
                     (df["uey"] >= y_bins[j]) & (df["uey"] < y_bins[j + 1])]
        
        # Calculate the average NMSE for the cell
        if not in_cell.empty:
            grid[i, j] = -in_cell["nmse"].mean()
        else:
            grid[i, j] = np.nan  # Assign NaN if no UEs fall into the cell

cmap = mpl.colormaps['viridis']  # Choose a colormap
im = ax.imshow(grid, origin="lower", extent=(y_bins[0], y_bins[-1], x_bins[0], x_bins[-1]), cmap=cmap)

# Add a colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Average SNDR")

ax.set_xlabel("Y Position [m]")
ax.set_ylabel("X Position [m]")

file_path = PurePath(dataset_path, "flickering/sndr_grid.pdf")
fig.savefig(file_path)

In [ ]:
file_path = PurePath(dataset_path, "flickering/results.csv")
df = pd.read_csv(file_path)

fig, ax = plt.subplots()

cmap = mpl.colormaps['viridis']  # Choose a colormap
nmse = -df["nmse"]
pc = ax.scatter(df["uey"], df["uex"], c=nmse, cmap=cmap)
ax.set_xlabel("Y Position [m]")
ax.set_ylabel("X Position [m]")

# Now plot all the RU positions on our figure.
x0 = stripe_config['stripe_start_pos'][0]
xend = stripe_config['stripe_end_pos'][0]
dx = stripe_config['space_between_stripes']
y0 = stripe_config['stripe_start_pos'][1]
yend = stripe_config['stripe_end_pos'][1]
dy = stripe_config['space_between_RUs']
n_rus = stripe_config['N_RUs']
n_stripes = stripe_config['N_stripes']

# Compute the RU positions.
x_pos = np.linspace(x0, xend, n_rus)
y_pos = np.linspace(y0, yend, n_rus)

# Repeat x_pos n_rus times, adding dx to each repetition.
x_pos_repeated = np.array([x_pos + i * dx for i in range(n_stripes)])
y_pos_repeated = np.array([y_pos for i in range(n_stripes)])

ax.scatter(y_pos_repeated, x_pos_repeated, s=5.0, color='red')

# Add a colorbar
cbar = fig.colorbar(pc, ax=ax)
cbar.set_label("SNDR [dB]")

file_path = PurePath(dataset_path, "flickering/sndr_map.pdf")
fig.savefig(file_path)

In [ ]:
ue_ds = xr.load_dataset(f"wireless_channel/sionna_dataset/{environment}/ue_locations/ue_locations.nc")

for i in range(6):
    user = ue_ds.where(ue_ds["user_id"] == i, drop=True)
    x = float(user["x"])
    y = float(user["y"])
    z = float(user["z"])

    # Read in the data for UEx
    df = pd.read_pickle(f"wireless_channel/sionna_dataset/{environment}/flickering/flickering_data_{i}.pkl")

    # Make a plot for every UE. The plot is a heatmap with the data for every stripe on the y-axis and the data for every
    # RU on the x-axis.

    for rub_id in df["ru_beam_id"].unique():
        for ueb_id in df["ue_beam_id"].unique():
            # Get the data for this specific stripe.
            beam_data = df.loc[(df["ru_beam_id"] == rub_id) & (df["ue_beam_id"] == ueb_id)]
            data = beam_data.pivot(index="stripe_id", columns="ru_id", values="pavg_ru")
            #for col in data.columns:
            #    data[col] = [a.mean() for a in data[col]]
            #data = 10 * np.log10(data)
        
            fig, ax = plt.subplots()
            im = ax.imshow(data)
            startx = stripe_config["stripe_start_pos"][0]
            starty = stripe_config["stripe_start_pos"][1]
            circle = Circle(((y - starty), (x - startx)), radius=0.2, edgecolor='red', facecolor='none', linewidth=2)
            ax.add_patch(circle)

            # Create colorbar
            cbar = ax.figure.colorbar(im, ax=ax)
            cbar.ax.set_ylabel("SNDR [dB]", rotation=-90, va="bottom")

            ax.set_title(f"UE: {i}, rub: {rub_id}, ueb: {ueb_id}")
            ax.set_ylabel("Stripe")
            ax.set_xlabel("Radio Unit")